# Experimentos

In [1]:
# Limita o BLAS/OpenMP a 1 thread por processo ANTES de importar numpy/sklearn.
# Como paralelizamos no nivel de tasks (ProcessPoolExecutor com fork), nao queremos
# cada worker abrindo seu proprio pool de threads do BLAS (super-subscricao de CPU).
# Tambem evita a interacao problematica entre fork e threads internas do BLAS, que e
# a causa comum de o kernel travar/corromper ("module 'numpy' has no attribute ...").
# Nao altera resultados: as matrizes do MLP sao pequenas (BLAS ja roda ~single-thread).
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")
os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "1")

import numpy as np
import random
import time
import pandas as pd

# Importando Problemas
from src.problems.DTLZ import DTLZ1, DTLZ2, DTLZ3, DTLZ4

# Importando as classes para o uso do DVL
from src.dvl.models.MLP import MLPModel
from src.dvl.models.SVR import SVRModel
# from src.dvl.models.Linear import LinearModel
from src.dvl.DVLMOEAWrapper import DVLMOEAWrapper

# Importando Indicadores de Qualidade
from src.QualityIndicator import HV

# Importando MOEAs
from src.MOEAs.MOEAD import MOEAD
from src.MOEAs.NSGAII import NSGAII
from src.MOEAs.NSGAIII import NSGAIII
from src.MOEAs.mutations.PolynomialMutation import PolynomialMutation
from src.MOEAs.crossovers.SBXCrossover import SBXCrossover
from src.BinaryTournament import BinaryTournament
from src.MOEAs.sparsities.CrowdingDistance import CrowdingDistance

MODEL_CLASSES = {
    "DTLZ1": SVRModel,
    "DTLZ2": MLPModel,
    "DTLZ3": SVRModel,
    "DTLZ4": MLPModel,
}

# Pontos de referencia para o calculo do Hypervolume conforme a Tabela 17 da
# dissertacao (experimento de otimizacao do DVL Framework).
HV_REFERENCE_POINTS = {
    ("DTLZ1", 3): np.array([1.0, 1.0, 1.0], dtype=float),
    ("DTLZ1", 10): np.array([5.0] * 10, dtype=float),
    ("DTLZ2", 3): np.array([2.0, 2.0, 2.0], dtype=float),
    ("DTLZ2", 10): np.array([1.5] * 10, dtype=float),
    ("DTLZ3", 3): np.array([10.0, 10.0, 10.0], dtype=float),
    ("DTLZ3", 10): np.array([50.0] * 10, dtype=float),
    ("DTLZ4", 3): np.array([2.0, 2.0, 2.0], dtype=float),
    ("DTLZ4", 10): np.array([1.5] * 10, dtype=float),
}

# Teto de amostras usadas para TREINAR o modelo inverso por ponto de referencia.
# Regra unica "treinar em min(amostra_LHS, MAX_TRAINING_SAMPLES)":
#   - NO-OP para budgets baixos (e<=1000, amostra < 500) -> resultados identicos
#     aos ja salvos; podem ser mantidos.
#   - Corta o custo dominante (treino ~99% do tempo) em e=1500(dvl_50/75), e=10000
#     e 100000, sem perder qualidade da frente (validado: HV igual ou melhor).
# IMPORTANTE: NAO usamos seed_moea_with_samples aqui. Aquele recurso mudaria a
# populacao inicial do MOEA em TODAS as execucoes (inclusive e<=1000 ja feitas),
# quebrando a continuidade com os resultados existentes. O cap de treino, nao.
MAX_TRAINING_SAMPLES = 500

# Variacoes avaliadas: MOEA puro (baseline) + DVL gastando 25/50/75% do
# orcamento total de avaliacoes. dvl_fraction=0.0 marca o baseline.
VARIATIONS = [
    {"name": "pure_moea", "mode": "pure_moea",     "dvl_fraction": 0.0},
    {"name": "dvl_25",    "mode": "dvl_framework", "dvl_fraction": 0.25},
    {"name": "dvl_50",    "mode": "dvl_framework", "dvl_fraction": 0.50},
    {"name": "dvl_75",    "mode": "dvl_framework", "dvl_fraction": 0.75},
]


In [2]:
def run_with_dvl(
    problem_class,    # Classe do problema
    moea_class,       # Classe do algoritmo evolucionario
    model_class,      # Classe do modelo de ML do DVL (MLP)
    m,                # Numero de objetivos (M)
    k,                # Parametro k do DTLZ
    max_evaluations,  # Numero total de avaliacoes (DVL + MOEA)
    dvl_fraction,     # Fracao das avaliacoes gastas no DVL (0.10, 0.25, 0.50, 0.75)
    seed=42,
):
    """Executa o pipeline DVL(X%) -> MOEA via DVLMOEAWrapper.

    Retorna (population, wrapper, problem). O wrapper expoe as metricas de
    avaliacoes (dvl_evaluations / moea_evaluations) e de tempo (model_training_time).
    """
    np.random.seed(seed)
    random.seed(seed)

    problem = problem_class(numberOfObjectives=m, k=k)
    # MLP reproduz a configuracao 'MLPSS' do Artur (StandardScaler + lbfgs).
    if model_class.__name__ == "MLPModel":
        model = model_class(solver="lbfgs")
    else:
        model = model_class()

    crossover = SBXCrossover(30.0, 1.0)
    mutation_probability = 1.0 / problem.numberOfDecisionVariables
    mutation = PolynomialMutation(mutation_probability, 20.0)

    wrapper = DVLMOEAWrapper(
        problem=problem,
        moea_class=moea_class,
        model=model,
        m=m,
        max_evaluations=max_evaluations,
        dvl_fraction=dvl_fraction,
        sampling_seed=seed,
        # IMPORTANTE: sem transformacao de objetivos (ver notas de reproducao).
        objective_transform=None,
        crossover=crossover,
        mutation=mutation,
        # Cap de treino: treina em min(amostra, MAX_TRAINING_SAMPLES). No-op para
        # e<=1000; corta o custo nos budgets altos sem perder qualidade. Sem
        # seed_moea_with_samples (manteria continuidade com os resultados antigos).
        max_training_samples=MAX_TRAINING_SAMPLES,
    )

    population = wrapper.execute()
    return population, wrapper, problem


In [3]:
def run_without_dvl(
    problem_class,  # Classe do problema (ex: DTLZ1, DTLZ2)
    moea_class,     # Classe do algoritmo evolucionário (ex: NSGAIII, NSGAII)
    m,              # Número de objetivos do problema (M)
    k,              # Parâmetro k do problema DTLZ
    max_evaluations,# Número máximo de avaliações
    seed=42
):
    import inspect
    from src.Util import ReferencePoint

    np.random.seed(seed)
    random.seed(seed)

    problem = problem_class(numberOfObjectives=m, k=k)
    
    div_dict = {2: 99, 3: 12, 10: 3}
    num_div = div_dict.get(m, 12)
    
    crossover = SBXCrossover(30.0, 1.0)
    mutation_probability = 1.0 / problem.numberOfDecisionVariables
    mutation = PolynomialMutation(mutation_probability, 20.0)
    selection = BinaryTournament()
    sparsity = CrowdingDistance()
    
    # Gera pontos de referência para deduzir tamanho de população se o algoritmo precisar
    ref_points = ReferencePoint().generateReferencePoints(m, num_div)
    pop_size = len(ref_points)

    # Identificar assinatura do construtor dinamicamente
    sig = inspect.signature(moea_class)
    params = sig.parameters
    
    kwargs = {}
    if "problem" in params:
        kwargs["problem"] = problem
    if "maxEvaluations" in params:
        kwargs["maxEvaluations"] = max_evaluations
    if "crossover" in params:
        kwargs["crossover"] = crossover
    if "mutation" in params:
        kwargs["mutation"] = mutation
    if "selection" in params:
        kwargs["selection"] = selection
    if "sparsity" in params:
        kwargs["sparsity"] = sparsity
    if "numberOfDivisions" in params:
        kwargs["numberOfDivisions"] = num_div
    if "populationSize" in params:
        kwargs["populationSize"] = pop_size
    if "offSpringPopulationSize" in params:
        kwargs["offSpringPopulationSize"] = pop_size if pop_size % 2 == 0 else pop_size + 1
        
    moea = moea_class(**kwargs)
    population = moea.execute()
    if population is None:
        population = moea.population
    
    return population, problem


In [4]:
def build_experiment_configs():
    budgets = [250, 500, 1_000, 1_500, 10_000, 100_000]

    # (M, k): 3 objetivos -> k=10 (n=12); 10 objetivos -> k=3 (n=12).
    objectives = [(3, 10), (10, 3)]
    problems = ["DTLZ1", "DTLZ2", "DTLZ3", "DTLZ4"]

    configs = []
    for m, k in objectives:
        for problem_name in problems:
            for e in budgets:
                configs.append({
                    "problem_name": problem_name,
                    "m": m,
                    "problem_k": k,
                    "max_evaluations": e,
                    "label": f"{problem_name}_m{m}_e{e}",
                })
    return configs


In [5]:
def run_experiment(config, problem_class, algo_class, variation, seed):
    """Executa um unico experimento (config x algoritmo x variacao x seed),
    persiste a populacao final em CSV e retorna o dicionario de resultado.

    NOTA: a escrita das metricas em out/all_results.csv NAO acontece mais aqui.
    Com execucao paralela (ProcessPoolExecutor) varios workers nao podem escrever
    no mesmo CSV ao mesmo tempo (corromperia o arquivo). O processo principal e
    o unico escritor das metricas (ver a celula do loop principal). Cada CSV de
    populacao tem caminho unico por task, entao a escrita aqui e segura em paralelo.

    Determinismo: cada task re-semeia np.random/random no inicio (em run_with_dvl/
    run_without_dvl) e o MLP usa esse estado global -> o resultado e funcao apenas
    do seed, identico independente da ordem ou do paralelismo."""
    label = config["label"]
    problem_name = config["problem_name"]
    m = config["m"]
    k = config["problem_k"]
    max_evals = config["max_evaluations"]

    var_name = variation["name"]
    mode = variation["mode"]
    dvl_fraction = variation["dvl_fraction"]

    result = {
        "label": label,
        "problem": problem_name,
        "algorithm": algo_class.__name__,
        "mode": mode,
        "variation": var_name,
        "dvl_fraction": dvl_fraction,
        "m": m,
        "problem_k": k,
        "max_evaluations": max_evals,
        "seed": seed,
        "cpu_time_seconds": 0.0,       # tempo total da execucao
        "model_training_time": 0.0,    # tempo de treino do modelo (DVL)
        "real_evaluation_time": 0.0,   # tempo gasto na funcao objetivo real
        "objective_calls": 0,          # total de avaliacoes reais consumidas
        "dvl_evaluations": 0,          # avaliacoes gastas no DVL
        "moea_evaluations": 0,         # avaliacoes gastas no MOEA
        "sample_size": 0,              # amostra LHS planejada (DVL)
        "estimated_points": 0,         # pontos estimados planejados (DVL)
        "population_size_final": None,
        "hypervolume": 0.0,
        "status": "success",
        "error_message": "",
    }

    population = None
    start_time = time.perf_counter()
    try:
        if mode == "pure_moea":
            population, problem = run_without_dvl(
                problem_class=problem_class,
                moea_class=algo_class,
                m=m,
                k=k,
                max_evaluations=max_evals,
                seed=seed,
            )
            result["model_training_time"] = 0.0
            result["real_evaluation_time"] = problem.evaluation_time
            result["objective_calls"] = problem.avaliations
            result["dvl_evaluations"] = 0
            result["moea_evaluations"] = problem.avaliations
        elif mode == "dvl_framework":
            population, wrapper, problem = run_with_dvl(
                problem_class=problem_class,
                moea_class=algo_class,
                model_class=MODEL_CLASSES[problem_name],
                m=m,
                k=k,
                max_evaluations=max_evals,
                dvl_fraction=dvl_fraction,
                seed=seed,
            )
            result["model_training_time"] = wrapper.model_training_time
            result["real_evaluation_time"] = wrapper.real_evaluation_time
            result["objective_calls"] = wrapper.objective_calls
            result["dvl_evaluations"] = wrapper.dvl_evaluations
            result["moea_evaluations"] = wrapper.moea_evaluations
            result["sample_size"] = wrapper.planned_sample_size
            result["estimated_points"] = wrapper.planned_estimated_points
        else:
            raise ValueError(f"Mode desconhecido: {mode}")

        end_time = time.perf_counter()
        result["cpu_time_seconds"] = end_time - start_time
        result["population_size_final"] = len(population)

        # Hypervolume normalizado pelo volume do ponto de referencia.
        ref_point = HV_REFERENCE_POINTS.get((problem_name, m))
        if ref_point is not None:
            objectives_list = [sol.objectives for sol in population]  # type: ignore
            indicator = HV(referencePoint=ref_point)
            hv_val = indicator.calculate(objectives_list)
            result["hypervolume"] = hv_val / np.prod(ref_point)
    except Exception as e:
        end_time = time.perf_counter()
        result["cpu_time_seconds"] = end_time - start_time
        result["status"] = "error"
        result["error_message"] = str(e)
        result["hypervolume"] = np.nan

    os.makedirs("out", exist_ok=True)

    # Persistir a populacao final (objetivos + variaveis de decisao).
    # Caminho unico por task -> seguro mesmo com varios workers em paralelo.
    if result["status"] == "success" and population is not None:
        pop_data = []
        for sol in population:
            row = {}
            for idx, obj in enumerate(sol.objectives):  # type: ignore
                row[f"obj_{idx}"] = obj
            for idx, var in enumerate(sol.decisionVariables):  # type: ignore
                row[f"var_{idx}"] = var
            pop_data.append(row)
        pop_df = pd.DataFrame(pop_data)
        pop_csv_path = (
            f"out/{label}_{algo_class.__name__}_{var_name}_seed_{seed}_population.csv"
        )
        pop_df.to_csv(pop_csv_path, index=False)

    return result


In [6]:
# Loop principal de experimentos (PARALELIZADO via ProcessPoolExecutor)
#
# Por que paralelizar nao altera os resultados:
#   - Cada task re-semeia np.random/random (em run_with_dvl/run_without_dvl) e o
#     MLPRegressor (random_state=None) usa esse estado global -> o resultado de
#     uma task e funcao deterministica APENAS do seed.
#   - Cada task roda em um PROCESSO separado (isolamento total de estado). Logo o
#     resultado independe da ordem e de quantos workers rodam em paralelo.
#   - O processo principal e o UNICO escritor de out/all_results.csv (evita corrida
#     de escrita). Os CSVs de populacao tem caminho unico por task (escritos pelos
#     workers, sem corrida).
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor, as_completed

configs = build_experiment_configs()

problem_classes = {
    "DTLZ1": DTLZ1,
    "DTLZ2": DTLZ2,
    "DTLZ3": DTLZ3,
    "DTLZ4": DTLZ4,
}
algorithms = [MOEAD, NSGAII, NSGAIII]

# Configuracoes de execucao
SEED = 42
NUM_RUNS = 20
results = []

# Carrega execucoes anteriores (mesmo schema) para nao reexecuta-las (resume).
existing_runs = set()
result_columns = None
csv_path = "out/all_results.csv"
if os.path.exists(csv_path):
    try:
        df_existing = pd.read_csv(csv_path)
        if "variation" in df_existing.columns:
            result_columns = list(df_existing.columns)
            results = df_existing.to_dict(orient="records")
            for r in results:
                key = (
                    str(r["label"]),
                    str(r["algorithm"]),
                    str(r["variation"]),
                    int(r["seed"]),
                )
                existing_runs.add(key)
            print(f"Carregados {len(results)} registros anteriores de {csv_path}.")
        else:
            print(
                f"AVISO: {csv_path} tem schema antigo (sem coluna 'variation'). "
                "Apague a pasta out/ antes de re-executar."
            )
    except Exception as e:
        print(f"Erro ao carregar execucoes anteriores: {e}")

total_runs = len(configs) * len(algorithms) * NUM_RUNS * len(VARIATIONS)

# Monta a lista de tarefas pendentes (pula as ja concluidas).
tasks = []
for config in configs:
    problem_class = problem_classes.get(config["problem_name"])
    if not problem_class:
        continue
    for algo_class in algorithms:
        for i in range(NUM_RUNS):
            seed = SEED + i
            for variation in VARIATIONS:
                key = (
                    str(config["label"]),
                    str(algo_class.__name__),
                    str(variation["name"]),
                    int(seed),
                )
                if key in existing_runs:
                    continue
                tasks.append((config, problem_class, algo_class, variation, seed))

print(
    f"Total: {total_runs} | ja concluidas: {total_runs - len(tasks)} | "
    f"pendentes: {len(tasks)}\n"
)


def _init_worker():
    """Limita BLAS/OpenMP a 1 thread por processo para evitar super-subscricao
    de CPU (N workers x M threads internas). Nao muda o resultado da otimizacao
    (que e governado pelo seed); se algo, deixa o BLAS deterministico."""
    try:
        from threadpoolctl import threadpool_limits
        threadpool_limits(1)
    except Exception:
        pass


def _append_metrics(res, metrics_csv_path, columns):
    """Escreve uma linha de metricas no CSV (chamado SO pelo processo principal).
    Reindexa para a ordem de colunas canonica para nunca desalinhar o arquivo."""
    row_df = pd.DataFrame([res])
    if columns is not None:
        row_df = row_df.reindex(columns=columns)
    row_df.to_csv(
        metrics_csv_path,
        mode="a",
        header=not os.path.exists(metrics_csv_path),
        index=False,
    )


if tasks:
    os.makedirs("out", exist_ok=True)
    metrics_csv_path = "out/all_results.csv"
    n_workers = max(1, (os.cpu_count() or 2) - 1)
    ctx = mp.get_context("fork")  # fork: workers herdam funcoes/globais do notebook

    print(f"Iniciando pool com {n_workers} workers...\n")
    start_time = time.perf_counter()
    completed = 0
    with ProcessPoolExecutor(
        max_workers=n_workers, mp_context=ctx, initializer=_init_worker
    ) as executor:
        future_to_key = {
            executor.submit(run_experiment, cfg, pcls, acls, var, sd):
                (cfg["label"], acls.__name__, var["name"], sd)
            for (cfg, pcls, acls, var, sd) in tasks
        }
        for future in as_completed(future_to_key):
            label, algo_name, var_name, sd = future_to_key[future]
            try:
                res = future.result()
            except Exception as e:
                # Falha catastrofica do worker (ex.: processo morto). run_experiment
                # ja captura excecoes internas, entao isto e raro.
                res = {
                    "label": label, "algorithm": algo_name, "variation": var_name,
                    "seed": sd, "status": "error", "error_message": str(e),
                    "hypervolume": np.nan,
                }
            results.append(res)

            # Inicializa a ordem de colunas no primeiro resultado (caso CSV novo).
            if result_columns is None:
                result_columns = list(res.keys())
            _append_metrics(res, metrics_csv_path, result_columns)

            completed += 1
            if completed % 20 == 0 or completed == len(tasks):
                elapsed = time.perf_counter() - start_time
                rate = elapsed / completed
                remaining = rate * (len(tasks) - completed)
                print(
                    f"Progresso: {completed}/{len(tasks)} | "
                    f"decorrido {elapsed/60:.1f}min | restante ~{remaining/60:.1f}min"
                )

print("\nExperimentos concluidos.")


/home/gustavo/Documents/pesquisa/maop/PyManyobjective/.venv/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:545: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)
/home/gustavo/Documents/pesquisa/maop/PyManyobjective/.venv/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:545: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)
/home/gustavo/Documents/pesquisa/maop/PyManyobjective/.venv/lib/python3.10/site-pack

Evaluations: 0 de 25000...


/home/gustavo/Documents/pesquisa/maop/PyManyobjective/.venv/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:545: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)
/home/gustavo/Documents/pesquisa/maop/PyManyobjective/.venv/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:545: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)
/home/gustavo/Documents/pesquisa/maop/PyManyobjective/.venv/lib/python3.10/site-pack

Evaluations: 0 de 75000...


/home/gustavo/Documents/pesquisa/maop/PyManyobjective/.venv/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:545: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)
/home/gustavo/Documents/pesquisa/maop/PyManyobjective/.venv/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:545: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)
/home/gustavo/Documents/pesquisa/maop/PyManyobjective/.venv/lib/python3.10/site-pack

...

Evaluations: 0 de 75000...


/home/gustavo/Documents/pesquisa/maop/PyManyobjective/.venv/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:545: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)
/home/gustavo/Documents/pesquisa/maop/PyManyobjective/.venv/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:545: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)
/home/gustavo/Documents/pesquisa/maop/PyManyobjective/.venv/lib/python3.10/site-pack

Evaluations: 0 de 50000...


/home/gustavo/Documents/pesquisa/maop/PyManyobjective/.venv/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:545: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)
/home/gustavo/Documents/pesquisa/maop/PyManyobjective/.venv/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:545: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)
/home/gustavo/Documents/pesquisa/maop/PyManyobjective/.venv/lib/python3.10/site-pack

Evaluations: 0 de 25000...
Progresso: 6136/6136 | decorrido 860.0min | restante ~0.0min

Experimentos concluidos.


In [7]:
# Visualizacao dos resultados brutos das execucoes
df = pd.DataFrame(results)

df_pure = df[df["variation"] == "pure_moea"]
df_dvl = df[df["variation"] != "pure_moea"]

print("Execucoes MOEA puro:")
display(df_pure)

print("\nExecucoes MOEA + DVL (25/50/75%):")
display(df_dvl)


Execucoes MOEA puro:


,label,problem,algorithm,mode,variation,dvl_fraction,m,problem_k,max_evaluations,seed,...,real_evaluation_time,objective_calls,dvl_evaluations,moea_evaluations,sample_size,estimated_points,population_size_final,hypervolume,status,error_message
0,DTLZ1_m3_e250,DTLZ1,MOEAD,pure_moea,pure_moea,0.0,3,10,250,42,...,0.007840,250,0,250,0,0,22,0.00000,success,NaN
4,DTLZ1_m3_e250,DTLZ1,MOEAD,pure_moea,pure_moea,0.0,3,10,250,43,...,0.007735,250,0,250,0,0,20,0.00000,success,NaN
8,DTLZ1_m3_e250,DTLZ1,MOEAD,pure_moea,pure_moea,0.0,3,10,250,44,...,0.006897,250,0,250,0,0,22,0.00000,success,NaN
12,DTLZ1_m3_e250,DTLZ1,MOEAD,pure_moea,pure_moea,0.0,3,10,250,45,...,0.006638,250,0,250,0,0,22,0.00000,success,NaN
16,DTLZ1_m3_e250,DTLZ1,MOEAD,pure_moea,pure_moea,0.0,3,10,250,46,...,0.006682,250,0,250,0,0,27,0.00000,success,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11502,DTLZ4_m10_e100000,DTLZ4,NSGAIII,pure_moea,pure_moea,0.0,10,3,100000,57,...,4.638793,100000,0,100000,0,0,188,1.00000,success,
11504,DTLZ4_m10_e100000,DTLZ4,NSGAIII,pure_moea,pure_moea,0.0,10,3,100000,58,...,4.410851,100000,0,100000,0,0,164,1.00000,success,
11510,DTLZ4_m10_e100000,DTLZ4,NSGAIII,pure_moea,pure_moea,0.0,10,3,100000,59,...,4.616467,100000,0,100000,0,0,220,0.99846,success,
11514,DTLZ4_m10_e100000,DTLZ4,NSGAIII,pure_moea,pure_moea,0.0,10,3,100000,60,...,3.843174,100000,0,100000,0,0,220,0.99863,success,



Execucoes MOEA + DVL (25/50/75%):


,label,problem,algorithm,mode,variation,dvl_fraction,m,problem_k,max_evaluations,seed,...,real_evaluation_time,objective_calls,dvl_evaluations,moea_evaluations,sample_size,estimated_points,population_size_final,hypervolume,status,error_message
1,DTLZ1_m3_e250,DTLZ1,MOEAD,dvl_framework,dvl_25,0.25,3,10,250,42,...,0.011183,250,62,188,31,31,22,0.000000,success,NaN
2,DTLZ1_m3_e250,DTLZ1,MOEAD,dvl_framework,dvl_50,0.50,3,10,250,42,...,0.007860,250,125,125,63,62,23,0.000000,success,NaN
3,DTLZ1_m3_e250,DTLZ1,MOEAD,dvl_framework,dvl_75,0.75,3,10,250,42,...,0.007816,278,188,90,97,91,91,0.056812,success,NaN
5,DTLZ1_m3_e250,DTLZ1,MOEAD,dvl_framework,dvl_25,0.25,3,10,250,43,...,0.007040,250,62,188,31,31,18,0.000000,success,NaN
6,DTLZ1_m3_e250,DTLZ1,MOEAD,dvl_framework,dvl_50,0.50,3,10,250,43,...,0.006273,250,125,125,63,62,26,0.235530,success,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11513,DTLZ4_m10_e100000,DTLZ4,NSGAIII,dvl_framework,dvl_50,0.50,10,3,100000,60,...,4.336057,100000,50000,50000,49780,220,220,0.998360,success,
11515,DTLZ4_m10_e100000,DTLZ4,NSGAIII,dvl_framework,dvl_75,0.75,10,3,100000,61,...,4.228210,100000,75000,25000,74780,220,220,0.998440,success,
11516,DTLZ4_m10_e100000,DTLZ4,NSGAIII,dvl_framework,dvl_25,0.25,10,3,100000,60,...,3.261303,100000,25000,75000,24780,220,220,0.998690,success,
11518,DTLZ4_m10_e100000,DTLZ4,NSGAIII,dvl_framework,dvl_50,0.50,10,3,100000,61,...,3.475545,100000,50000,50000,49780,220,220,0.998410,success,


In [8]:
import os
import pandas as pd

pd.set_option("display.float_format", "{:.6f}".format)

csv_path = "out/all_results.csv"
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)

if not df.empty and "hypervolume" in df.columns:
    summary_df = df.groupby(
        ["problem", "m", "max_evaluations", "algorithm", "variation"]
    ).agg(
        hv_mean=("hypervolume", "mean"),
        hv_std=("hypervolume", "std"),
        cpu_time_mean=("cpu_time_seconds", "mean"),
        cpu_time_std=("cpu_time_seconds", "std"),
        model_training_time_mean=("model_training_time", "mean"),
        real_eval_time_mean=("real_evaluation_time", "mean"),
        dvl_evals_mean=("dvl_evaluations", "mean"),
        moea_evals_mean=("moea_evaluations", "mean"),
        objective_calls_mean=("objective_calls", "mean"),
        successful_runs=("status", lambda x: (x == "success").sum()),
    ).reset_index()

    # Ordena variacoes numa ordem logica (puro -> 25 -> 50 -> 75).
    var_order = ["pure_moea", "dvl_25", "dvl_50", "dvl_75"]
    summary_df["variation"] = pd.Categorical(
        summary_df["variation"], categories=var_order, ordered=True
    )
    summary_df = summary_df.sort_values(
        by=["problem", "m", "algorithm", "max_evaluations", "variation"]
    )

    print("Resumo estatistico (media por configuracao):")
    display(summary_df)
else:
    print("Nenhum resultado disponivel para resumir.")


Resumo estatistico (media por configuracao):


,problem,m,max_evaluations,algorithm,variation,hv_mean,hv_std,cpu_time_mean,cpu_time_std,model_training_time_mean,real_eval_time_mean,dvl_evals_mean,moea_evals_mean,objective_calls_mean,successful_runs
3,DTLZ1,3,250,MOEAD,pure_moea,0.000000,0.000000,0.047036,0.004411,0.000000,0.006766,0.000000,250.000000,250.000000,20
0,DTLZ1,3,250,MOEAD,dvl_25,0.000000,0.000000,0.262943,0.030635,0.182744,0.006737,62.000000,188.000000,250.000000,20
1,DTLZ1,3,250,MOEAD,dvl_50,0.041101,0.068800,0.545506,0.041090,0.428049,0.006378,125.000000,125.000000,250.000000,20
2,DTLZ1,3,250,MOEAD,dvl_75,0.297584,0.103892,0.916440,0.033168,0.757339,0.006840,188.000000,90.000000,278.000000,20
15,DTLZ1,3,500,MOEAD,pure_moea,0.000000,0.000000,0.096032,0.007743,0.000000,0.013523,0.000000,500.000000,500.000000,20
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
562,DTLZ4,10,10000,NSGAIII,dvl_75,0.997588,0.000419,191.288012,2.097513,174.482964,0.487622,7500.000000,2500.000000,10000.000000,20
575,DTLZ4,10,100000,NSGAIII,pure_moea,0.999012,0.000914,598.876400,61.372138,0.000000,4.560410,0.000000,100000.000000,100000.000000,20
572,DTLZ4,10,100000,NSGAIII,dvl_25,0.998600,0.000141,638.558888,51.409486,157.854620,4.460228,25000.000000,75000.000000,100000.000000,20
573,DTLZ4,10,100000,NSGAIII,dvl_50,0.998541,0.000133,481.732771,32.240905,156.790042,4.536737,50000.000000,50000.000000,100000.000000,20
